In [1]:
import psycopg2
import pandas as pd

In [2]:
# environment variable can take two values: DEV (development), PRD (production)
env = 'DEV'

In [3]:
# for production we will use following database uri variable
DATABASE_URI = ''

In [5]:
try:
    if env == 'DEV':
        con = psycopg2.connect(host="localhost", user='postgres', database='value-investing-dev', port='5432', password='v,1846PSVv,1846PSV')
    elif env == 'PRD':
        con = psycopg2.connect(DATABASE_URI)

    #create cursor to execute sql statements
    cur = con.cursor()

    # read the tax rates from the damodaran file
    df = pd.read_csv('CountryRisks.csv')

    # iterate through dataframe to create new entries or update existing ones
    for index, row in df.iterrows():
        #extract country
        country = str(row['Country'])

        # extract moody rating
        moody_rating = row["Moody's rating"]

        # extract country risk; check if % sign is included
        country_risk = row["Country Risk Premium"]
        if '%' in country_risk:
            country_risk = float(country_risk.split('%')[0])/100
        else:
            country_risk = float(country_risk)/100

        # first we will check if already a country with the same name exists in the database
        sql_select_query = """select * from public.dcf_countryrisk where lower(country) = %s"""

        # execute the sql query
        cur.execute(sql_select_query, (country.lower(),))

        countries = cur.fetchall()

        # if length is zero (so there does not yet exist and entry in the table), we will create a new entry; otherwise we will update the existing one
        if len(countries) == 0:
            sql_insert_query = """INSERT INTO public.dcf_countryrisk(country, "countryRisk", "moodyRating") VALUES(%s, %s, %s)"""
            cur.execute(sql_insert_query, (country, country_risk, moody_rating))
            print(f'inserted into table: Country: {country}, CountryRisk {country_risk}, Moody rating {moody_rating}')
        else:
            sql_update_query = """UPDATE public.dcf_countryrisk SET "countryRisk"=%s, "moodyRating"=%s where lower(country)=%s"""
            cur.execute(sql_update_query, (country_risk, moody_rating, country.lower()))
            print(f'updated in table: Country: {country}, CountryRisk {country_risk}, Moody rating {moody_rating}')
        
        # commit
        con.commit()
    
    cur.close()

except Exception as error:
    print('Could not connect to the database: ', error)


updated in table: Country: Abu Dhabi, CountryRisk 0.0085, Moody rating Aa2
updated in table: Country: Albania, CountryRisk 0.07769999999999999, Moody rating B1
updated in table: Country: Andorra (Principality of), CountryRisk 0.0329, Moody rating Baa2
updated in table: Country: Angola, CountryRisk 0.11220000000000001, Moody rating B3
updated in table: Country: Argentina, CountryRisk 0.2071, Moody rating Ca
updated in table: Country: Armenia, CountryRisk 0.0621, Moody rating Ba3
updated in table: Country: Aruba, CountryRisk 0.0329, Moody rating Baa2
updated in table: Country: Australia, CountryRisk 0.0, Moody rating Aaa
updated in table: Country: Austria, CountryRisk 0.0069, Moody rating Aa1
updated in table: Country: Azerbaijan, CountryRisk 0.0432, Moody rating Ba1
updated in table: Country: Bahamas, CountryRisk 0.07769999999999999, Moody rating B1
updated in table: Country: Bahrain, CountryRisk 0.0949, Moody rating B2
updated in table: Country: Bangladesh, CountryRisk 0.0621, Moody ra